## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

1. **Critical Care Protocols:** "What is the protocol for managing sepsis in a critical care unit?"

2. **General Surgery:** "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"

3. **Dermatology:** "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"

4. **Neurology:** "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"


### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [66]:
# Install required libraries
!pip install -q langchain_community==0.3.27 \
              langchain==0.3.27 \
              chromadb==1.0.15 \
              pymupdf==1.26.3 \
              tiktoken==0.9.0 \
              datasets==4.0.0 \
              evaluate==0.4.5 \
              langchain_openai==0.3.30

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [67]:
# Import core libraries
import os
import json
import requests  # type: ignore

# Import libraries for working with PDFs and OpenAI
from langchain.document_loaders import PyMuPDFLoader
from langchain_community.document_loaders import PyPDFLoader
from openai import OpenAI                                                       # Access OpenAI

# Import libraries for processing dataframes and text
import tiktoken                                                                 # Tokenizer
import pandas as pd                                                             # Load, manipulate, and analyze data

# Import LangChain components for data loading, chunking, embedding, and vector DBs
from langchain.text_splitter import RecursiveCharacterTextSplitter              # Break text into overlapping chunks
from langchain.embeddings.openai import OpenAIEmbeddings                        # Create vector embeddings
from langchain.vectorstores import Chroma

from datasets import Dataset
from langchain_openai import ChatOpenAI

## Question Answering using LLM

> **Note 1:** When choosing between an open-source Hugging Face (HF) model and OpenAI’s proprietary model, base your decision on your specific needs. If you opt for a Hugging Face model, make sure to connect to a GPU to execute the code efficiently.

> **Note 2**: If the free-tier GPU of Google Colab is not accessible (due to unavailability or exhaustion of daily limit or other reasons), the following steps can be taken:
1. Wait for 12-24 hours until the GPU is accessible again or the daily usage limits are reset.
2. Switch to a different Google account and resume working on the project from there.
3. Try using the CPU runtime:
    - To use the CPU runtime, click on *Runtime* => *Change runtime type* => *CPU* => *Save*
    - One can also click on the *Continue without GPU* option to switch to a CPU runtime (kindly refer to the snapshot below)
    - The instructions for running the code on the CPU are provided in the respective sections of the notebook.

#### Downloading and Loading the model

In [68]:
# Load the JSON file and extract values
file_name = "config.json"                                                       # Name of the configuration file
with open(file_name, 'r') as file:                                              # Open the config file in read mode
    config = json.load(file)                                                    # Load the JSON content as a dictionary
    OPENAI_API_KEY = config.get("OPENAI_API_KEY")                                             # Extract the API key from the config
    OPENAI_API_BASE = config.get("OPENAI_API_BASE")                             # Extract the OpenAI base URL from the config

# Store API credentials in environment variables
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY                                          # Set API key as environment variable
os.environ["OPENAI_BASE_URL"] = OPENAI_API_BASE                                 # Set API base URL as environment variable

# Initialize OpenAI client
client = OpenAI()                                                               # Create an instance of the OpenAI client

In [69]:
import json

config_data = {
    "OPENAI_API_KEY": "gl-U2FsdGVkX1/KpXlBVke1Y2QQZeoG4GQs/0LzN1EsI/UIAhlMirO4JD9GRzENx6Mc",
    "OPENAI_API_BASE": "https://aibe.mygreatlearning.com/openai/v1"
}

with open("config.json", "w") as f:
    json.dump(config_data, f, indent=4)

#print("config.json created. Please update it with your actual API key and base URL.")

In [70]:
# Define a function to get a response
def response(user_prompt, max_tokens=512, temperature=0.75, top_p=0.95):   # set paramenters
    # Create a chat completion using the OpenAI client
    completion = client.chat.completions.create(
        model="gpt-4o-mini",                                                  # Specify the model to use
        messages=[
            {"role": "user", "content": user_prompt}                            # User prompt is the input/query to respond to
        ],
        max_tokens=max_tokens,                                                  # Max number of tokens to generate in the response
        temperature=temperature,                                                # Controls randomness in output
        top_p=top_p                                                             # Controls diversity via nucleus sampling
    )
    return completion.choices[0].message.content

### Question 1: What is the protocol for managing sepsis in a critical care unit?

In [71]:
question_1 = "What is the protocol for managing sepsis in a critical care unit?"
base_prompt_response_1=response(question_1)

### Question 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [72]:
question_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
base_prompt_response_2=response(question_2)

### Question 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [73]:
question_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
base_prompt_response_3=response(question_3)

### Question 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [74]:
question_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
base_prompt_response_4=response(question_4)

Create a DataBase

In [75]:
# Create the DataFrame
result_df = pd.DataFrame({
    "questions": [question_1, question_2, question_3, question_4],
    "base_prompt_responses": [base_prompt_response_1, base_prompt_response_2, base_prompt_response_3, base_prompt_response_4]})

# Display the DataFrame
result_df.head()

,questions,base_prompt_responses
0,What is the protocol for managing sepsis in a ...,Managing sepsis in a critical care unit involv...
1,"What are the common symptoms for appendicitis,...",Common symptoms of appendicitis include:\n\n1....
2,What are the effective treatments or solutions...,"Sudden patchy hair loss, often referred to as ..."
3,What treatments are recommended for a person w...,Treatment for a person who has sustained a phy...


Observation:Question answering using LLM


*   All four questions have been answered in depth using Open Ai model(LLM)
*   Next is to compare these with results generated using prompt engineering



## Question Answering using LLM with Prompt Engineering

In [76]:
system_prompt = """
You are a highly knowledgeable and experienced medical assistant. Your task is to provide accurate, concise, and helpful information to medical professionals based on your vast medical knowledge. When answering questions, focus on clarity and practical relevance. Ensure your responses are informative and directly address the query without unnecessary conversational filler.

"""

In [101]:
# Define a function to get a response from the OpenAI chat model
def response(system_prompt, user_prompt, max_tokens=512, temperature=0.2, top_p=0.95):  #set default paramenters
    # Create a chat completion using the OpenAI client
    completion = client.chat.completions.create(
        model="gpt-4o-mini",                                                        # the model to be used.
        messages=[
            {"role": "system", "content": system_prompt},                       # System prompt sets the assistant's behavior
            {"role": "user", "content": user_prompt}                            # User prompt is the input/query to respond to
        ],
        max_tokens=max_tokens,                                                  # Max number of tokens to generate in the response
        temperature=temperature,                                                # Controls randomness in output (0 = deterministic)
        top_p=top_p                                                             # Controls diversity via nucleus sampling
    )
    return completion.choices[0].message.content

### Question 1: What is the protocol for managing sepsis in a critical care unit?

In [78]:
response_with_prompt_eng_1=response(system_prompt,question_1)
response_with_prompt_eng_1

"The management of sepsis in a critical care unit follows a structured approach, often guided by the Surviving Sepsis Campaign guidelines. Here’s a concise protocol:\n\n1. **Early Recognition**:\n   - Monitor for signs of sepsis: fever, tachycardia, tachypnea, altered mental status, and hypotension.\n   - Use the Sequential Organ Failure Assessment (SOFA) score to assess organ dysfunction.\n\n2. **Immediate Resuscitation**:\n   - Administer broad-spectrum intravenous (IV) antibiotics within the first hour of recognition.\n   - Initiate fluid resuscitation with crystalloids (e.g., normal saline or lactated Ringer's solution) to achieve a target mean arterial pressure (MAP) of ≥65 mmHg.\n   - Consider vasopressors (e.g., norepinephrine) if hypotension persists despite adequate fluid resuscitation.\n\n3. **Source Control**:\n   - Identify and control the source of infection (e.g., drainage of abscess, removal of infected devices).\n\n4. **Monitoring**:\n   - Continuously monitor vital sig

### Question 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [79]:
response_with_prompt_eng_2=response(system_prompt,question_2)
response_with_prompt_eng_2

'Common symptoms of appendicitis include:\n\n1. **Abdominal Pain**: Typically starts around the navel and then shifts to the lower right abdomen.\n2. **Nausea and Vomiting**: Often follows the onset of abdominal pain.\n3. **Loss of Appetite**: Patients may not feel like eating.\n4. **Fever**: Usually low-grade but can increase as the condition progresses.\n5. **Constipation or Diarrhea**: Some patients may experience changes in bowel habits.\n6. **Abdominal Swelling**: In some cases, the abdomen may appear distended.\n\nAppendicitis cannot be effectively treated with medication alone; it typically requires surgical intervention. The standard procedure for treating appendicitis is an **appendectomy**, which involves the surgical removal of the appendix. This can be performed via:\n\n- **Open Appendectomy**: A larger incision in the abdomen.\n- **Laparoscopic Appendectomy**: A minimally invasive technique using small incisions and a camera.\n\nLaparoscopic appendectomy is often preferred

### Question 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [80]:
response_with_prompt_eng_3=response(system_prompt,question_3) #
response_with_prompt_eng_3

'Sudden patchy hair loss, often referred to as alopecia areata, can manifest as localized bald spots on the scalp. Here are effective treatments and potential causes:\n\n### Treatments:\n1. **Topical Corticosteroids**: These are often the first line of treatment. They help reduce inflammation and promote hair regrowth.\n   \n2. **Minoxidil (Rogaine)**: This over-the-counter topical solution can stimulate hair growth and is sometimes used in conjunction with corticosteroids.\n\n3. **Intralesional Corticosteroid Injections**: Administered directly into the bald patches, these can be effective for larger areas or more severe cases.\n\n4. **Immunotherapy**: For extensive alopecia areata, treatments like diphencyprone (DPCP) can induce an allergic reaction that may stimulate hair regrowth.\n\n5. **Oral Corticosteroids**: In cases of severe or widespread hair loss, a short course of oral corticosteroids may be prescribed.\n\n6. **Anthralin**: A topical medication that can help in some cases 

### Question 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [81]:
response_with_prompt_eng_4=response(system_prompt,question_4)
response_with_prompt_eng_4

'Treatment for a person who has sustained a physical injury to brain tissue, such as a traumatic brain injury (TBI), varies based on the severity of the injury and the specific impairments experienced. Here are common treatment approaches:\n\n1. **Emergency Care**: Immediate assessment and stabilization are crucial. This may involve:\n   - Monitoring vital signs\n   - Imaging studies (CT or MRI) to assess the extent of the injury\n   - Surgical intervention if there is significant bleeding or pressure on the brain\n\n2. **Medical Management**:\n   - **Medications**: To manage symptoms such as pain, seizures, or swelling (e.g., corticosteroids).\n   - **Neuroprotective agents**: Research is ongoing, but some agents may help protect brain tissue.\n\n3. **Rehabilitation**:\n   - **Physical Therapy**: To improve mobility and strength.\n   - **Occupational Therapy**: To assist with daily living activities and cognitive rehabilitation.\n   - **Speech Therapy**: For those with communication o

Adding all results from LLM + Prompt engineering

In [82]:
# Add the results to a new column in the DataFrame
result_df['responses_with_prompt_eng'] = [response_with_prompt_eng_1, response_with_prompt_eng_2, response_with_prompt_eng_3, response_with_prompt_eng_4]

# Display the DataFrame
result_df.head()

,questions,base_prompt_responses,responses_with_prompt_eng
0,What is the protocol for managing sepsis in a ...,Managing sepsis in a critical care unit involv...,The management of sepsis in a critical care un...
1,"What are the common symptoms for appendicitis,...",Common symptoms of appendicitis include:\n\n1....,Common symptoms of appendicitis include:\n\n1....
2,What are the effective treatments or solutions...,"Sudden patchy hair loss, often referred to as ...","Sudden patchy hair loss, often referred to as ..."
3,What treatments are recommended for a person w...,Treatment for a person who has sustained a phy...,Treatment for a person who has sustained a phy...


Observations from comparing, reults from LLMs and LLMs and prompts

* When reducing the temparature from 0.75 to 0.2, the answers are more accurate than those generated previously with base LLM
* All four problem statements (questions) have been answered in depth using the OpenAI model (LLM).
*   Responses generated both with and without prompt engineering are now available in the DataFrame.
* The next logical step is to compare these response with results genrated using RAG.






## Data Preparation for RAG

### Loading the Data

In [83]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [84]:
manual_pdf_path = "/content/medical_diagnosis_manual.pdf"
pdf_loader = PyMuPDFLoader(manual_pdf_path)
manual = pdf_loader.load()

### Data Overview

#### Checking the first 3 pages

In [85]:
for i in range(3):
    print(f"Page Number : {i+1}",end="\n")
    print(manual[i].page_content,end="\n")

Page Number : 1
safia.parveen09@gmail.com
3LNJUQ109B
This file is meant for personal use by safia.parveen09@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 2
safia.parveen09@gmail.com
3LNJUQ109B
This file is meant for personal use by safia.parveen09@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 3
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    ...............................................................................................................

### Data Chunking

In [86]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=256,
    chunk_overlap= 75 #define the chunk overlap
)

In [87]:
document_chunks = pdf_loader.load_and_split(text_splitter)

len(document_chunks) #check the number of chunks

18834

The data has been split into chuncks of 256 with an overlap of 75 and the total number of chunks in the data is 18834

### Embedding

In [88]:
# Initialize the OpenAI Embeddings model with API credentials
embedding_model = OpenAIEmbeddings(
    openai_api_key=OPENAI_API_KEY,                                                     # Your OpenAI API key for authentication
    openai_api_base=OPENAI_API_BASE                                             # The OpenAI API base URL endpoint
)

# Generate embeddings (vector representations) for the first two document chunks
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)      # Embedding for chunk 0
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)      # Embedding for chunk 1

# Check and print the dimension (length) of the embedding vector
print("Dimension of the embedding vector ", len(embedding_1))

# Verify if both embeddings have the same dimension (should be True)
len(embedding_1) == len(embedding_2)

# Return/display the two embedding vectors for further inspection or use
embedding_1, embedding_2

Dimension of the embedding vector  1536


([-0.0014319060914546886,
  -0.0062404469248117525,
  -0.00510491179065447,
  -0.018461599085231203,
  -0.022644096351639786,
  0.023310100062050487,
  -0.029250845037780637,
  -0.019540523307957098,
  -0.02108565087302858,
  -0.025121626876111948,
  0.04043969358910554,
  -0.005511173811861114,
  0.022830576943519924,
  0.01791547708577577,
  0.006866489444022281,
  0.012807234885413743,
  0.01880792071662153,
  -0.02869140326214023,
  0.005441243124244751,
  -0.00309358321154396,
  -0.010942427103967135,
  0.023669741469625784,
  -0.013340037108684203,
  0.017928796861960743,
  -0.02709299659232885,
  -0.016929793158989945,
  0.008951078654795398,
  -0.022977098206845136,
  0.00457210956738401,
  -0.020166565827167623,
  -0.009783581430163522,
  -0.007132890555657511,
  -0.009190839282738054,
  -0.012933775553138871,
  -0.014692023262460439,
  0.01775563604626558,
  0.023270138870850315,
  0.009190839282738054,
  0.03345998185655917,
  -0.004375638677703832,
  0.005308042568427136,
 

### Vector Database

In [121]:
out_dir = 'chroma_db'   #name of the vectore Db

if not os.path.exists(out_dir):
  os.makedirs(out_dir)

# Building the vector store and saving it to disk for future use
vectorstore = Chroma.from_documents(
    document_chunks,                                                            # Documents to index
    embedding_model,                                                            # Embedding model for converting text to vectors
    persist_directory=out_dir                                                   # Save vector DB files here
)

vectorstore = Chroma(
    persist_directory=out_dir,
    embedding_function=embedding_model
)

vectorstore.embeddings



OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x78430fcdd2b0>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x7842decc6cf0>, model='text-embedding-ada-002', deployment='text-embedding-ada-002', openai_api_version='', openai_api_base='https://aibe.mygreatlearning.com/openai/v1', openai_api_type='', openai_proxy='', embedding_ctx_length=8191, openai_api_key='gl-U2FsdGVkX1/KpXlBVke1Y2QQZeoG4GQs/0LzN1EsI/UIAhlMirO4JD9GRzENx6Mc', openai_organization=None, allowed_special=set(), disallowed_special='all', chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None)

### Retriever

In [122]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3} #for each query, the retriever will return the top 3 most similar document chunks.
)

In [123]:
qna_system_message = """
You are a helpful medical assistant. Your task is to answer the user's question only using information from the provided medical manual excerpts. Do not use any outside knowledge. If the answer is not present in the given context, state that you cannot answer the question based on the provided information.
"""
qna_user_message_template = """
Context: {context}

Question: {question}

Answer:
"""

### Response Function

In [124]:
def generate_rag_response(user_input,k=3,max_tokens=512,temperature=0.2,top_p=0.95):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    # Generate the response
    try:
        response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": qna_system_message},
            {"role": "user", "content": user_message}
        ],
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p
        )
        # Extract and print the generated text from the response
        response = response.choices[0].message.content.strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## Question Answering using RAG

### Question 1: What is the protocol for managing sepsis in a critical care unit?

In [125]:
response_with_rag_1 = generate_rag_response(question_1)
response_with_rag_1

'The protocol for managing sepsis in a critical care unit includes the following steps:\n\n1. **Specimen Collection**: Parenteral antibiotics should be given after specimens of blood, body fluids, and wound sites have been taken for Gram stain and culture.\n\n2. **Empiric Therapy**: Very prompt empiric therapy should be started immediately after suspecting sepsis, as it is essential and may be lifesaving.\n\n3. **Antibiotic Selection**: The selection of antibiotics requires an educated guess based on:\n   - The suspected source of infection\n   - The clinical setting\n   - Knowledge or suspicion of causative organisms\n   - Sensitivity patterns common to that specific inpatient unit\n   - Previous culture results\n\n4. **Regimen for Septic Shock of Unknown Cause**: One regimen includes:\n   - Gentamicin or tobramycin 5.1 mg/kg IV once/day plus a 3rd-generation cephalosporin (options include cefotaxime 2 g q 6 to 8 h, ceftriaxone 2 g once/day, or ceftazidime 2 g IV q 8 h if Pseudomonas 

### Question 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [126]:
response_with_rag_2 = generate_rag_response(question_2)
response_with_rag_2

'The common symptoms for appendicitis include abdominal pain, anorexia, and abdominal tenderness. Appendicitis cannot be cured via medicine; the treatment is surgical removal of the appendix.'

### Question 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [127]:
response_with_rag_3 = generate_rag_response(question_3)
response_with_rag_3

'The effective treatments for sudden patchy hair loss, known as alopecia areata, include topical corticosteroids, intralesional corticosteroids, or, in severe cases, systemic corticosteroids. Other treatment options may include topical minoxidil, topical anthralin, topical immunotherapy (such as diphencyprone or squaric acid dibutylester), or psoralen plus ultraviolet A (PUVA). \n\nThe possible causes behind alopecia areata are not specified in the provided information, but it is noted that it occurs in people with no obvious skin or systemic disorder.'

### Question 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [128]:
response_with_rag_4 = generate_rag_response(question_4)
response_with_rag_4

'Initial treatment for a person who has sustained a physical injury to brain tissue includes ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure. Surgery may be needed in patients with more severe injury to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas. In the first few days after the injury, it is important to maintain adequate brain perfusion and oxygenation and to prevent complications of altered sensorium. Subsequently, many patients require rehabilitation.'

# Add the results to a new column in the DataFrame

In [129]:
result_df['responses_with_RAG'] = [response_with_rag_1, response_with_rag_2, response_with_rag_3, response_with_rag_4]

In [130]:
# Display the DataFrame
result_df.head()

,questions,base_prompt_responses,responses_with_prompt_eng,responses_with_RAG
0,What is the protocol for managing sepsis in a ...,Managing sepsis in a critical care unit involv...,The management of sepsis in a critical care un...,The protocol for managing sepsis in a critical...
1,"What are the common symptoms for appendicitis,...",Common symptoms of appendicitis include:\n\n1....,Common symptoms of appendicitis include:\n\n1....,The common symptoms for appendicitis include a...
2,What are the effective treatments or solutions...,"Sudden patchy hair loss, often referred to as ...","Sudden patchy hair loss, often referred to as ...",The effective treatments for sudden patchy hai...
3,What treatments are recommended for a person w...,Treatment for a person who has sustained a phy...,Treatment for a person who has sustained a phy...,Initial treatment for a person who has sustain...


Observation after using RAGs:

*  Base Prompt Responses: These answers are generally comprehensive, providing broad information that leverages the LLM's vast general knowledge. They are well-structured but might lack the specific, curated focus of a medical manual.

*   Prompt Engineering Responses: Similar to the base prompt, but the system prompt (You are a highly knowledgeable and experienced medical assistant...) appears to have guided the LLM to deliver more structured, clear, and practically relevant information, as intended.

* RAG Responses: These responses are notably more concise and highly specific, directly focusing on details that would likely be found in a specialized medical manual. This indicates the RAG system is effectively retrieving and utilizing the provided medical_diagnosis_manual.pdf context.


## Output Evaluation

In [131]:
groundedness_rater_system_message = """
You are an expert, strict at evaluating if an answer is grounded in the provided context.
Your task is to determine whether the given 'Answer' is explicitly supported by the 'Context from medical references'.

Strict scoring criteria:
- Score 5: The answer uses details, exact dosages, named protocols, and clinical thresholds that are directly found in the context.
- Score 4: Most specific claims in the answer are directly from the context, with only few details not found in context.
- Score 3: The answer lacks specific details from the context, or includes information not present in the context.
- Score 2: The answer has some overlap with context topics but most specific claims cannot be traced to the context.
- Score 1: The answer has little or no overlap with the specific information in the context.

Important: Only information that can be directly traced to the provided context counts as high score. Even if the information is medically correct, if it isnt from the context source then score it low.

Format your response as:
Groundness: [score]
Rsasoning: [brief explanation].
"""
relevance_rater_system_message = """
You are an expert at evaluating the relevance of an answer to a given question.
Respond with a score from 1 to 5, where 1 means the answer is completely irrelevant to the question, and 5 means the answer directly and comprehensively addresses the question.

Respond with high score if the answer directly addresses the question and is supported by the context, and low score if it does not address the question or uses information outside the provided context.
Also, provide a brief justification for your score
Scoring criteria:
- Score 5: The context is highly precise and directly relevant to the question.
- Score 4: The context is mostly relevant with few unrelated information.
- Score 3: The context is partially relevant but contains a lot of unrelated information.
- Score 2: The context has very little relevance to the question.
- Score 1: The context is not relevant to the question at all.

Format your response as:
Context Precision: [score]
Reasoning: [brief explanation] """
user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""

In [132]:
def generate_ground_relevance_response(user_input,response,  max_tokens=512, temperature=0.2, top_p=0.95):  # set default paramenters
    global qna_user_message_template

    context_for_query = [doc.page_content for doc in retriever.get_relevant_documents(user_input, k=5)]

    # Combine user_prompt and system_message to create the prompt
    groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=response)}
                [/INST]"""

    # Combine user_prompt and system_message to create the prompt
    relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=response)}
                [/INST]"""

    response_1 = client.chat.completions.create(
            model="gpt-4o",   #
            messages=[
                {"role": "user", "content": groundedness_prompt}
                ],
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p
            )

    response_2 = client.chat.completions.create(
            model="gpt-4o",   #
            messages=[
                {"role": "user", "content": relevance_prompt}
                ],
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p
            )

    return response_1.choices[0].message.content,response_2.choices[0].message.content

#### **Evaluation 1: Base Prompt Response Evaluation**

In [133]:
# Question 1
ground,rel = generate_ground_relevance_response(user_input=result_df.questions[0], response=result_df.base_prompt_responses[0], max_tokens=516)
print(ground,end="\n\n")
print(rel)

Groundness: 1  
Reasoning: The answer provided does not directly reference or use specific details from the context. The context mentions specific antibiotic regimens and emerging therapies for sepsis, but the answer discusses general management protocols and guidelines from the Surviving Sepsis Campaign, which are not found in the context.

Context Precision: 3
Reasoning: The context provided focuses on the administration of antibiotics and some emerging therapies for sepsis, but it does not comprehensively cover the full protocol for managing sepsis in a critical care unit. The answer, however, outlines a detailed protocol including early identification, resuscitation, antibiotic therapy, source control, and other supportive measures, which are not fully supported by the context. Therefore, the context is only partially relevant to the question.


In [134]:
# Question 2
ground,rel = generate_ground_relevance_response(user_input=result_df.questions[1], response=result_df.base_prompt_responses[1], max_tokens=516)  #
print(ground,end="\n\n")
print(rel)

Groundness: 3

Reasoning: The answer correctly identifies abdominal pain, anorexia (loss of appetite), and abdominal tenderness as symptoms of appendicitis, which are directly supported by the context. However, the additional symptoms listed (nausea, vomiting, fever, constipation, diarrhea, and abdominal swelling) are not mentioned in the provided context. The context states that treatment is surgical removal, which aligns with the answer's mention of an appendectomy. However, the context does not discuss the possibility of using antibiotics or the details about laparoscopic surgery, which are included in the answer. Therefore, while some information is supported, several specific claims are not directly found in the context.

Context Precision: 5  
Reasoning: The answer directly addresses the question by listing common symptoms of appendicitis and explaining that it typically requires surgical intervention, specifically an appendectomy, which is supported by the context provided. The 

In [135]:
# Question 3
ground,rel = generate_ground_relevance_response(user_input=result_df.questions[2], response=result_df.base_prompt_responses[2], max_tokens=516)  #
print(ground,end="\n\n")
print(rel)
print()
# Question 4
ground,rel = generate_ground_relevance_response(user_input=result_df.questions[3], response=result_df.base_prompt_responses[3], max_tokens=516)  #
print(ground,end="\n\n")
print(rel)

Groundness: 3  
Reasoning: The answer includes some treatments and causes that are mentioned in the context, such as topical corticosteroids, minoxidil, intralesional corticosteroid injections, and immunotherapy. However, it also introduces several causes and treatments not explicitly found in the context, such as genetic predisposition, stress, hormonal changes, nutritional deficiencies, infections, oral medications like methotrexate and cyclosporine, light therapy, hair transplantation, and stress management. These additional details are not directly supported by the provided context.

Context Precision: 5
Reasoning: The answer directly addresses the question by providing a comprehensive list of possible causes and effective treatments for sudden patchy hair loss, specifically alopecia areata. The context supports the answer with detailed information on treatments like topical corticosteroids, minoxidil, and immunotherapy, which are mentioned in the context. Additionally, the context

#### **Evaluation 2: Prompt Engineering Response Evaluation**

In [136]:
# Question 1
ground,rel = generate_ground_relevance_response(user_input=result_df.questions[0], response=result_df.responses_with_prompt_eng[0], max_tokens=516)
print(ground,end="\n\n")
print(rel)

# Question 1
ground,rel = generate_ground_relevance_response(user_input=result_df.questions[1], response=result_df.responses_with_prompt_eng[1], max_tokens=516)
print(ground,end="\n\n")
print(rel)

# Question 3
ground,rel = generate_ground_relevance_response(user_input=result_df.questions[2], response=result_df.responses_with_prompt_eng[2], max_tokens=516)
print(ground,end="\n\n")
print(rel)

# Question 4
ground,rel = generate_ground_relevance_response(user_input=result_df.questions[3], response=result_df.responses_with_prompt_eng[3], max_tokens=516)
print(rel)

Groundness: 1  
Reasoning: The answer provided outlines a detailed protocol for managing sepsis, including specific interventions and monitoring strategies. However, none of these details, such as the use of the Surviving Sepsis Campaign guidelines, SOFA score, fluid resuscitation targets, or specific supportive care measures, are mentioned in the provided context. The context focuses on antibiotic regimens and some emerging therapies, but does not cover the comprehensive protocol described in the answer.

Context Precision: 3
Reasoning: The context provided focuses primarily on the administration of antibiotics and some emerging therapies for sepsis, which are relevant to the management of sepsis. However, the answer includes a comprehensive protocol for managing sepsis that goes beyond the context, such as early recognition, resuscitation, source control, and supportive care, which are not mentioned in the context. Thus, while the answer is relevant, it is only partially supported by

#### **Evaluation 3: RAG Response Evaluation**

In [137]:
# Question 1
ground,rel = generate_ground_relevance_response(user_input=result_df.questions[0], response=result_df.responses_with_RAG[0], max_tokens=516)
print(ground,end="\n\n")
print(rel)

# Question 2
ground,rel = generate_ground_relevance_response(user_input=result_df.questions[1], response=result_df.responses_with_RAG[1], max_tokens=516)
print(ground,end="\n\n")
print(rel)

# Question 3
ground,rel = generate_ground_relevance_response(user_input=result_df.questions[2], response=result_df.responses_with_RAG[2], max_tokens=516)
print(ground,end="\n\n")
print(rel)

# Question 4
ground,rel = generate_ground_relevance_response(user_input=result_df.questions[3], response=result_df.responses_with_RAG[3], max_tokens=516)
print(ground,end="\n\n")
print(rel)

Groundness: 5  
Reasoning: The answer is directly supported by the context provided. It includes specific details such as the steps for specimen collection, the importance of prompt empiric therapy, criteria for antibiotic selection, and detailed regimens for septic shock, including dosages and options for antibiotics. All these elements are explicitly mentioned in the context.

Context Precision: 5  
Reasoning: The answer directly and comprehensively addresses the question about the protocol for managing sepsis in a critical care unit. It outlines the steps involved, including specimen collection, empiric therapy, and antibiotic selection, all of which are supported by the provided context. The details about specific antibiotic regimens for septic shock of unknown cause are also included, making the answer highly precise and relevant.
Groundness: 5  
Reasoning: The answer is fully supported by the context. It accurately lists the symptoms of appendicitis as abdominal pain, anorexia, a

## Actionable Insights and Business Recommendations



-The model performs significantly better when paired with a RAG system compared to using only an LLM with prompt engineering, as retrieval grounding improves factual accuracy, completeness, and reduces hallucination.



<font size=6 color='#4682B4'>Power Ahead</font>
___